# Week 7 / 8 - Decision Trees + Naive Bayes Classification + Model Comparison

## A. Setup + Data Preparation

In [ ]:
import sys
import platform
from pathlib import Path

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('Pandas:', pd.__version__)

Python: 3.13.1
Platform: Windows-10-10.0.19045-SP0
Pandas: 3.0.1


In [2]:
PROJECT_ROOT = Path.cwd() / "no_show_predict"
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS = PROJECT_ROOT / "reports"
CONFIGS = PROJECT_ROOT / "configs"

PROJECT_ROOT

WindowsPath('c:/Users/Vivi/Documents/Repos/Data-Science/no_show_predict')

In [6]:
data_path = DATA_PROCESSED / "appointments_processed.csv"
data = pd.read_csv(data_path)
print(data.dtypes)

PatientId         float64
AppointmentID       int64
Gender                str
ScheduledDay          str
AppointmentDay        str
Age                 int64
Neighbourhood         str
Scholarship          bool
Hypertension         bool
Diabetes             bool
Alcoholism           bool
Handicap             bool
SMSReceived          bool
NoShow               bool
WaitingDays         int64
dtype: object


In [22]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

y = data['NoShow'].astype(int)

X_orig = data.drop(columns=[
    'NoShow', 'PatientId', 'AppointmentID', 'ScheduledDay', 'AppointmentDay'
])

X_orig['Gender'] = (X_orig['Gender'] == 'F')
bool_cols = X_orig.select_dtypes(include='bool').columns.tolist()
for col in bool_cols:
    X_orig[col] = X_orig[col].astype(int)

cat_cols = ['Neighbourhood']
prep = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
    ]
)
X = prep.fit_transform(X_orig)

In [23]:
from sklearn.model_selection import train_test_split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print("Train set:", X_train.shape, y_train.shape)
print("Validation set:", X_val.shape, y_val.shape)
print("Test set:", X_test.shape, y_test.shape)

Train set: (77368, 80) (77368,)
Validation set: (16579, 80) (16579,)
Test set: (16580, 80) (16580,)


The chosen features are appropriate because ==AAA==

Scaling isn't really necessary since the only real numeric features are the patient's age, which have already been previously validated to be in a normal range.

`PatientId` and `AppointmentID` seem irrelevant because they are only used for internal tracking by the hospital, they don't mean anything and aren't even seen by a real patient.
`ScheduledDay` and `AppointmentDay` seem redundant since the derived column `WaitingDays` already exists.

**Selected validation criteria for comparison:**
- Precision
- Recall
- F1-Score

## B. Decision Tree Classification

### B1. Initial Decision Tree Model

In [34]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

model = DecisionTreeClassifier(
    max_depth = 5,
    min_samples_split = 10,
    min_samples_leaf = 5,
    random_state=42
)
model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_val_pred = model.predict(X_val)

train_acc = accuracy_score(y_train, y_train_pred)
val_acc = accuracy_score(y_val, y_val_pred)
precision = precision_score(y_val, y_val_pred, zero_division = 0)
recall = recall_score(y_val, y_val_pred, zero_division = 0)
f1 = f1_score(y_val, y_val_pred, zero_division = 0)
conf_mat = confusion_matrix(y_val, y_val_pred)

print("Model parameters:")
for param in model.get_params():
    print(f"\t-{param}: {model.get_params()[param]}")
print("Tree depth:", model.get_depth())
print("Number of leaves:", model.get_n_leaves())
print("Training accuracy:", train_acc)
print("Validation accuracy:", val_acc)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("Confusion Matrix:")
print(conf_mat)

Model parameters:
	-ccp_alpha: 0.0
	-class_weight: None
	-criterion: gini
	-max_depth: 5
	-max_features: None
	-max_leaf_nodes: None
	-min_impurity_decrease: 0.0
	-min_samples_leaf: 5
	-min_samples_split: 10
	-min_weight_fraction_leaf: 0.0
	-monotonic_cst: None
	-random_state: 42
	-splitter: best
Tree depth: 5
Number of leaves: 6
Training accuracy: 0.7972676041774377
Validation accuracy: 0.798057783943543
Precision: 0.0
Recall: 0.0
F1 Score: 0.0
Confusion Matrix:
[[13231     0]
 [ 3348     0]]


### B2. Controlling Tree Complexity

In [30]:
def train_tree_model(
        max_depth = None, 
        min_samples_split = 2, 
        min_samples_leaf = 1, 
        ccp_alpha = 0.0
):
    model = DecisionTreeClassifier(
        random_state=42,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        ccp_alpha=ccp_alpha
    )
    
    model.fit(X_train, y_train)
    return model

In [31]:
configs = [
    {"max_depth": 3, "min_samples_split": 2,  "min_samples_leaf": 1, "ccp_alpha": 0.0},
    {"max_depth": 5, "min_samples_split": 2,  "min_samples_leaf": 1, "ccp_alpha": 0.0},
    {"max_depth": 5, "min_samples_split": 10, "min_samples_leaf": 1, "ccp_alpha": 0.0},
    {"max_depth": 5, "min_samples_split": 10, "min_samples_leaf": 5, "ccp_alpha": 0.0},
    {"max_depth": 5, "min_samples_split": 10, "min_samples_leaf": 5, "ccp_alpha": 0.01},
]

In [36]:
results = []
for i, config in enumerate(configs, start = 1):
    model = train_tree_model(
        max_depth = config["max_depth"],
        min_samples_split = config["min_samples_split"],
        min_samples_leaf = config["min_samples_leaf"],
        ccp_alpha = config["ccp_alpha"]
    )

    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)

    train_acc = accuracy_score(y_train, y_train_pred)
    val_acc = accuracy_score(y_val, y_val_pred)
    precision = precision_score(y_val, y_val_pred, zero_division = 0)
    recall = recall_score(y_val, y_val_pred, zero_division = 0)
    f1 = f1_score(y_val, y_val_pred, zero_division = 0)

    results.append({
        "config_id": i,
        "max_depth": config["max_depth"],
        "min_samples_split": config["min_samples_split"],
        "min_samples_leaf": config["min_samples_leaf"],
        "ccp_alpha": config["ccp_alpha"],
        "tree_depth": model.get_depth(),
        "num_leaves": model.get_n_leaves(),
        "train_acc": train_acc,
        "val_acc": val_acc,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    })

    print(f"Config {i} confusion matrix:")
    print(confusion_matrix(y_val, y_val_pred))

results_df = pd.DataFrame(results)
print(results_df)

Config 1 confusion matrix:
[[13231     0]
 [ 3348     0]]
Config 2 confusion matrix:
[[13231     0]
 [ 3348     0]]
Config 3 confusion matrix:
[[13231     0]
 [ 3348     0]]
Config 4 confusion matrix:
[[13231     0]
 [ 3348     0]]
Config 5 confusion matrix:
[[13231     0]
 [ 3348     0]]
   config_id  max_depth  min_samples_split  min_samples_leaf  ccp_alpha  \
0          1          3                  2                 1       0.00   
1          2          5                  2                 1       0.00   
2          3          5                 10                 1       0.00   
3          4          5                 10                 5       0.00   
4          5          5                 10                 5       0.01   

   tree_depth  num_leaves  train_acc   val_acc  precision  recall  f1_score  
0           3           4   0.797268  0.798058        0.0     0.0       0.0  
1           5           6   0.797268  0.798058        0.0     0.0       0.0  
2           5           6

Create a suitable plot showing how model performance changes with tree complexity (for example: validation accuracy vs. max_depth).
Select the best Decision Tree model based on validation performance.
Write 1–2 sentences: Did increasing tree depth always improve generalization? Briefly relate your observation to underfitting / overfitting.

### B3. Interpretation of the Tree

For the best Decision Tree model, report:
-final depth
-number of leaves
-most important features
Visualize the tree, or at least the first few levels of the tree.
Extract and write 3–5 decision rules from the tree.
Write 2–3 sentences: Is the tree easy to interpret? Do the rules make sense for the dataset?

## C. Naive Bayes Classification

### C1. Choosing the Naive Bayes Model

Choose the appropriate Naive Bayes variant for your dataset, such as:
-GaussianNB for mainly continuous numerical features
-MultinomialNB for count-based features
-CategoricalNB if the features are categorical and encoded appropriately
Write 1–2 sentences: Why is this Naive Bayes variant appropriate/ not appropriate for your dataset?

### C2. Fit and Evaluate Naive Bayes

Train the Naive Bayes model on the same training data.
Report:
-model type used
-training accuracy
-validation accuracy
-other metrics
Show a confusion matrix for the Naive Bayes model.
If relevant, test at least 2–3 different preprocessing choices (for example: different encodings, feature subsets, or with/without scaling where reasonable).
Create a small summary table of the Naive Bayes experiments.

### C3. Interpretation of Naive Bayes Results

Briefly discuss the independence assumption of Naive Bayes.
Write 2–3 sentences: Do you think the feature independence assumption is realistic for your dataset? How might this affect performance?

## D. Final Comparison of Classification Models

### D1. Comparison Table

Create a final comparison table including:
-Model name
-Main parameter(s) used
    -Decision Tree: max_depth, min_samples_split, min_samples_leaf, or ccp_alpha
    -Naive Bayes: model type and important preprocessing choices
-Training accuracy
-Validation accuracy
-Test accuracy
-Other metrics

Compare:
-best Decision Tree model
-best Naive Bayes model

Identify which model performed best under each evaluation criterion.

### D2. Analysis of Results

**Did all evaluation criteria agree on the better model? If not, explain why.**

**Compare the 2 methods**:
- Predictive Performance
- Interpretability
- Eobustness
- Ease of Tuning

**Which model appears more sensitive to feature choice, feature dependence, or overfitting?**

## E. Final Prediction Check

Use your best final model to make predictions on the test set.
Show at least 5 example predictions with:
-true label
-predicted label
Briefly comment on:
-one correct prediction
-one incorrect prediction

**Final Answer**: Which classification method would you recommend for this dataset overall, and why?
- Classification Quality:
- Interpretability:
- Robustness:
- Assumptions of the Model:
- Practicality of Training & Tuning: